# Custom Tools Agent

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSV-AI/agent-playground/blob/dev/notebooks/custom_tools.ipynb)

## Description
Custom Tools Agent that provides various functionalities to assist users in performing tasks using specialized tools. The tools are implemented with custom functions that are annotated with Pydantic-ai plain_tool. There is no system prompt for this agent - it's only purpose is to demonstrate the use of custom developed tools.


## Overview

This notebook demonstrates how to create a custom tools agent using PydanticAI with OpenRouter as the LLM provider. The agent can:
- Get the current time
- Retrieve user information
- Access company logos (as images)
- Process PDF documents

## Setup

Before running this notebook, make sure you have the required dependencies installed and your OpenRouter API key configured.

In [1]:
# Install required dependencies
# Uncomment the line below if running in Google Colab or if dependencies are not installed
!pip install pydantic-ai pydantic python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 4.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-instrumentation-httpx to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.2/422.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [18]:
from pydantic_ai import Agent, RunContext, DocumentUrl, ImageUrl
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openrouter import OpenRouterProvider
from datetime import datetime
from pydantic import BaseModel

# Used to manage secrets
from google.colab import userdata


## Define Data Models

Define a Pydantic model for user data. For more information on Pydantic Data Models, see the [documentation](https://docs.pydantic.dev/latest/concepts/models/)

In [3]:
class User(BaseModel):
    name: str
    age: int

## Configure the LLM

Configure PydanticAI to use OpenRouter with OpenAI-compatible model. To use with Colab, add a secret (Click the Key Icon on the Left) for OPENROUTER_API_KEY

In [4]:
# --- Configuration ---
# Since OpenRouter has a unified API, we can use the OpenAIChatModel with a custom provider.
OPENROUTER_MODEL = "openai/gpt-4o-mini"
OPENROUTER_KEY = userdata.get('OPENROUTER_API_KEY')

# 1. Configure the LLM for OpenRouter
# We use OpenAIChatModel because OpenRouter is OpenAI-compatible
# and we pass the OpenRouterProvider to configure the endpoint.
openrouter_model = OpenAIChatModel(
    OPENROUTER_MODEL,
    provider=OpenRouterProvider(
        api_key=OPENROUTER_KEY
    ),
)

## Create the Agent

Initialize the agent with the configured LLM model.

In [30]:
# 2. Create the Agent with the WebSearchTool
agent = Agent(
    model=openrouter_model,
)

## Define Custom Tools

Define custom tools that the agent can use to answer user queries.

This is a little tricky to do inside of a Google Colab notebook. Because these tools are registered with the agents list of tools, you can't run this cell again without clearing the python kernel or running the cell above to create a new agent instance.

In [6]:
@agent.tool_plain
def get_current_time() -> datetime:
    print("get_current_time tool called")
    return datetime.now()

@agent.tool_plain
def get_user() -> User:
    print("get_user tool called")
    return User(name='John', age=30)

@agent.tool_plain
def get_company_logo() -> ImageUrl:
    print("get_company_logo tool called")
    return ImageUrl(url='https://iili.io/3Hs4FMg.png')

@agent.tool_plain
def get_document() -> DocumentUrl:
    print("get_document tool called")
    return DocumentUrl(url='https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf')

## Run the Agent

Test the agent with various queries. It will use the custom tools to provide answers.

**Note:** The asynchronous run method used with **await** is normally called within a function marked with **async**. In this case, the notebook is already running with an async/await context behind the scenes.

In [11]:
# Execute a query to get the current time
result = await agent.run('What time is it?')
print(result.output)

get_current_time tool called
The current time is 02:20 AM on November 30, 2025.


In [12]:
# Execute a query to get the user name
result = await agent.run('What is the user name?')
print(result.output)


get_user tool called
The user name is John.


In [14]:
# Execute a query to identify the company name from the logo
result = await agent.run('What is the company name in the logo?')
print(result.output)

get_company_logo tool called
The company name in the logo is "Pydantic."


In [15]:
result = await agent.run('What is the main content of the document?')
print(result.output)

get_document tool called
The document appears to contain the phrase "Dummy PDF file," indicating that it may not have substantial or meaningful content. If there are specific aspects or additional details you are looking for, please let me know!


# Additional Result Information

There is additional information available with the result along with the ouput of the model. For example, we can look through the list of tool calls, model calls, and get a count of the tokens used.

[AgentRunResult Documentation](https://ai.pydantic.dev/api/run/#pydantic_ai.run.AgentRunResult)

In [21]:
# Call the agent
result = await agent.run('What is the main content of the document?')

# Get the usage object by calling result.usage()
usage_info = result.usage()

print(f"Usage found: Input tokens={usage_info.input_tokens} | Output tokens={usage_info.output_tokens}")

get_document tool called
Usage found: Input tokens=415 | Output tokens=47


# Run Context

The tools that we have implemented so far did not use any additional information from the model. If there is information needed such as the prompt, or any previous turn-by-turn messages, we can add the Run Context to the tool call.

[Run Context Documentation](https://ai.pydantic.dev/api/tools/#pydantic_ai.tools.RunContext)

In [31]:
@agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """Get the player's name."""
    print(f'Prompt: {ctx.prompt}')
    print(f'Dependencies: {ctx.deps}')
    print('Messages:')
    for message in ctx.messages:
      print(f'    {message}')
    return "Harry"

In [32]:
# Call the agent
result = await agent.run("What is the player's name?")
print(result.output)


Prompt: What is the player's name?
Dependencies: None
Messages:
    ModelRequest(parts=[UserPromptPart(content="What is the player's name?", timestamp=datetime.datetime(2025, 11, 30, 3, 26, 52, 548106, tzinfo=datetime.timezone.utc))], run_id='021d13f2-d01b-414d-a2f9-2538c593428e')
    ModelResponse(parts=[ToolCallPart(tool_name='get_player_name', args='{}', tool_call_id='call_c83nVABdTIP8Xg7ZoqCYgZb3')], usage=RequestUsage(input_tokens=43, output_tokens=11, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}), model_name='openai/gpt-4o-mini', timestamp=datetime.datetime(2025, 11, 30, 3, 26, 52, tzinfo=TzInfo(0)), provider_name='openrouter', provider_details={'finish_reason': 'tool_calls'}, provider_response_id='gen-1764473212-howt2KowUyxeZgR6xARh', finish_reason='tool_call', run_id='021d13f2-d01b-414d-a2f9-2538c593428e')
The player's name is Harry.
